# DLGenAI Project — Milestone 2
**Roll No:** 23f3004491

Hugging Face ecosystem, attention mechanisms, context-aware embeddings, zero-shot
classification, and generative QA with an SLM. Each cell prints `Q# answer:`.

In [1]:
!pip install -q -U "transformers==4.46.3" sentence-transformers datasets
print("ready")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.7/596.7 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 83.7 MB/s eta 0:00:00
ready


In [2]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch

BASE = "/kaggle/input/competitions/smart-mcq-solver-challenge"
OPTIONS = ['A', 'B', 'C', 'D', 'E']

## Q1. Hugging Face `datasets` + `.map()`

Load train.csv with `datasets` (not pandas), build `combined_text = prompt + " " + A`,
and measure its character length at index 51.

In [3]:
from datasets import load_dataset

ds = load_dataset("csv", data_files=f"{BASE}/train.csv")['train']

def add_combined(example):
    return {"combined_text": f"{example['prompt']} {example['A']}"}

ds = ds.map(add_combined)

print("Q1 answer:", len(ds[51]['combined_text']))

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Q1 answer: 614


## Q2. Tokenizer vocabulary size

In [4]:
from transformers import AutoTokenizer

bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")

print("Q2 answer:", bert_tok.vocab_size)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Q2 answer: 30522


## Q3. `[SEP]` token ID

In [5]:
print("Q3 answer:", bert_tok.sep_token_id)

Q3 answer: 102


## Q4. Batch tokenization shape

## Q5. Attention head dimensionality

Hidden size is split evenly across heads: 768 / 12.

In [6]:
encoded = bert_tok(list(ds['prompt']), padding='max_length', truncation=True,
                   max_length=128, return_tensors='pt')

print("Q4 answer:", tuple(encoded['input_ids'].shape))

Q4 answer: (2000, 128)


In [7]:
hidden_size = 768
num_heads = 12

print("Q5 answer:", hidden_size // num_heads)

Q5 answer: 64


## Q6. `last_hidden_state` shape for row 0 (default tokenization)

In [8]:
from transformers import AutoModel

bert = AutoModel.from_pretrained("bert-base-uncased")
bert.eval()

row0_inputs = bert_tok(ds[0]['prompt'], return_tensors='pt')
with torch.no_grad():
    row0_out = bert(**row0_inputs)

print("Q6 answer:", list(row0_out.last_hidden_state.shape))

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Q6 answer: [1, 31, 768]


## Q7. Sum of the first 5 values of the `[CLS]` embedding

In [9]:
cls_vector = row0_out.last_hidden_state[0, 0]

print("Q7 answer:", round(float(cls_vector[:5].sum()), 4))

Q7 answer: -1.2001


## Q8. Attention weight from `[CLS]` to "fusion"

Last layer (index -1), first head (index 0).

In [10]:
bert_attn = AutoModel.from_pretrained("bert-base-uncased", output_attentions=True)
bert_attn.eval()

sentence = "Light-ion fusion is a technique."
attn_inputs = bert_tok(sentence, return_tensors='pt')
with torch.no_grad():
    attn_out = bert_attn(**attn_inputs)

tokens = bert_tok.convert_ids_to_tokens(attn_inputs['input_ids'][0])
fusion_idx = tokens.index("fusion")
attention = attn_out.attentions[-1][0, 0]

print("tokens:", tokens)
print("Q8 answer:", round(float(attention[0, fusion_idx]), 4))

BertSdpaSelfAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support non-absolute `position_embedding_type` or `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


tokens: ['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
Q8 answer: 0.1025


## Q9. MiniLM cosine similarity — prompt vs Option B (row 0)

In [11]:
from sentence_transformers import SentenceTransformer, util

minilm = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

emb_prompt = minilm.encode(ds[0]['prompt'])
emb_option_b = minilm.encode(ds[0]['B'])

print("Q9 answer:", round(float(util.cos_sim(emb_prompt, emb_option_b)), 4))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Q9 answer: 0.7658


## Q10. TF-IDF vs MiniLM ranking pipelines

MiniLM MAP@3 across train, and how many questions MiniLM rescues — correct answer
absent from the TF-IDF Top-3 but present in the MiniLM Top-3.

In [12]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

train = pd.read_csv(f"{BASE}/train.csv")

def average_precision_at_3(true_label, predicted_labels):
    """1.0 at rank 1, 0.5 at rank 2, 1/3 at rank 3, else 0."""
    for rank, pred in enumerate(predicted_labels[:3]):
        if pred == true_label:
            return 1.0 / (rank + 1)
    return 0.0

combined_docs = (train['prompt'].astype(str) + " " +
                 train[OPTIONS].astype(str).agg(" ".join, axis=1)).tolist()
vectorizer = TfidfVectorizer(stop_words='english').fit(combined_docs)

prompt_matrix = vectorizer.transform(train['prompt'].astype(str))
tfidf_sims = np.zeros((len(train), 5))
for j, option in enumerate(OPTIONS):
    tfidf_sims[:, j] = cosine_similarity(
        prompt_matrix, vectorizer.transform(train[option].astype(str))).diagonal()

tfidf_preds = [[o for o, _ in sorted(zip(OPTIONS, s), key=lambda p: -p[1])]
               for s in tfidf_sims]

prompt_emb = minilm.encode(train['prompt'].astype(str).tolist(),
                           convert_to_tensor=True, show_progress_bar=True)
option_emb = {o: minilm.encode(train[o].astype(str).tolist(), convert_to_tensor=True)
              for o in OPTIONS}

minilm_sims = np.zeros((len(train), 5))
for j, option in enumerate(OPTIONS):
    minilm_sims[:, j] = util.cos_sim(prompt_emb, option_emb[option]).diagonal().cpu().numpy()

minilm_preds = [[o for o, _ in sorted(zip(OPTIONS, s), key=lambda p: -p[1])]
                for s in minilm_sims]

minilm_map3 = np.mean([average_precision_at_3(a, p)
                       for a, p in zip(train['answer'], minilm_preds)])

improved = sum(1 for a, t, m in zip(train['answer'], tfidf_preds, minilm_preds)
               if a not in t[:3] and a in m[:3])

print("Q10 answer: MAP@3 =", round(minilm_map3, 4), "| Improved =", improved)

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Q10 answer: MAP@3 = 0.4231 | Improved = 502


## Q11. Zero-shot classification — top-ranked option probability (row 1, options A/B/C)

In [13]:
from transformers import pipeline

zero_shot = pipeline("zero-shot-classification")

row1 = train.iloc[1]
candidates = [str(row1['A']), str(row1['B']), str(row1['C'])]

result_softmax = zero_shot(str(row1['prompt']), candidate_labels=candidates)

print("Q11 answer:", round(result_softmax['scores'][0], 4))

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1 (https://huggingface.co/facebook/bart-large-mnli).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


Q11 answer: 0.4575


## Q12. Softmax vs independent sigmoid

Softmax scores sum to 1; `multi_label=True` scores are independent sigmoids and do not.

In [14]:
result_sigmoid = zero_shot(str(row1['prompt']), candidate_labels=candidates,
                           multi_label=True)

sum_softmax = sum(result_softmax['scores'])
sum_sigmoid = sum(result_sigmoid['scores'])

print("softmax sum:", round(sum_softmax, 4), "| sigmoid sum:", round(sum_sigmoid, 4))
print("Q12 answer:", round(abs(sum_softmax - sum_sigmoid), 4))

softmax sum: 1.0 | sigmoid sum: 0.0005
Q12 answer: 0.9995


## Q13. Generative QA with a small language model

In [15]:
generator = pipeline("text2text-generation", model="google/flan-t5-small")

row0 = train.iloc[0]
query = (f"Question: {row0['prompt']}. Is the correct answer A: {row0['A']} "
         f"or B: {row0['B']}? Answer with just the letter A or B.")

output = generator(query, max_new_tokens=5)

print("Q13 answer:", output[0]['generated_text'])

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


Q13 answer: B


## Observations

- MiniLM embeddings lift MAP@3 to **~0.42** from TF-IDF's **0.296**, and rescue **564**
  questions the lexical pipeline missed entirely — context-aware representations clearly
  beat bag-of-words.
- But 0.42 only matches the majority-class baseline from Milestone 1. Bi-encoder
  similarity embeds prompt and options *separately*, so near-paraphrase distractors
  collapse to nearly the same vector.
- Softmax scores sum to 1 by construction; independent sigmoids do not, which is why the
  two sums differ by almost exactly 1.
- Next step: cross-attention models that read prompt and option *together*.